# Alignment pipeline (debug, fixed)

Robust discovery of episodes_audio by scanning upward through parent directories. Prints cwd, chosen episodes dir and found .webm files.

In [4]:
from pathlib import Path
import warnings
from tqdm import TqdmWarning

# suppress tqdm widget warning in environments without ipywidgets
warnings.filterwarnings("ignore", category=TqdmWarning)

from audio_matcher.embedding import AudioEmbeddingPipeline
from audio_matcher.phonemes import PhonemeAligner
from audio_matcher.alignment import build_phoneme_index_from_episodes, run_phoneme_pipeline
from audio_matcher.io import export_audio

# target song (adjust if needed)
SONG_PATH = Path("../separated/htdemucs/audio/vocals.wav")


In [5]:
# Robust upward search for episodes_audio folder
repo_root = Path.cwd()
print('notebook cwd =', repo_root)
EPISODES_DIR = None
for p in [repo_root] + list(repo_root.parents):
    candidates = [
        p / 'data' / 'episodes_audio',
        p / 'audio_matcher' / 'data' / 'episodes_audio',
        p / 'episodes_audio',
    ]
    for c in candidates:
        if c.exists():
            EPISODES_DIR = c
            break
    if EPISODES_DIR is not None:
        break

if EPISODES_DIR is None:
    EPISODES_DIR = repo_root / 'audio_matcher' / 'data' / 'episodes_audio'

print('EPISODES_DIR chosen:', EPISODES_DIR)
if not EPISODES_DIR.exists():
    print('EPISODES_DIR does not exist:', EPISODES_DIR)
    files = []
else:
    files = sorted(EPISODES_DIR.rglob('audio.wav'))
    print(f'Found {len(files)} audio.wav files under {EPISODES_DIR}')
    for f in files:
        print('-', f)


notebook cwd = C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\notebooks
EPISODES_DIR chosen: C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio
Found 30 audio.wav files under C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio\1TlOcjJodHw\audio.wav
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio\326R_Lhua5w\audio.wav
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio\3l1lSNQxJA0\audio.wav
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio\6uNYmqvF24k\audio.wav
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio\9aLD8SGoPIc\audio.wav
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\

In [6]:
# Build phoneme index from all episodes (multi-file).
import gc
import torch
import numpy as np

pipeline = AudioEmbeddingPipeline()
aligner = PhonemeAligner(device='cpu', whisper_model='base')
files = files

pindex = build_phoneme_index_from_episodes(files, aligner, pipeline)


C:\Users\yihab\miniconda3\envs\video_analyzer\Lib\site-packages\transformers\configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [7]:
# final: run phoneme pipeline on chosen song and export
if not files:
    print('No reference files found, skipping pipeline')
else:
    final_audio = run_phoneme_pipeline(SONG_PATH, None, aligner, pipeline, pindex=pindex)
    print(f"output_ms: {len(final_audio)}")
    export_audio(final_audio, 'aligned_output.wav')
    print('Wrote aligned_output.wav')


NameError: name 'pindex' is not defined